# Cross-Encoder Reranking for RAG

## Overview

First-stage retrieval (bi-encoder + FAISS) is **fast** but **imprecise**. It maps queries and
documents into a shared vector space *independently*, so the similarity metric can only capture
surface-level overlap. In practice this means top-K results often include passages that share
vocabulary with the query but don't actually answer it.

**Reranking** closes this quality gap by inserting a second, more expensive model—a
**cross-encoder**—that reads the query *and* each candidate document jointly, allowing deep
token-level interaction via cross-attention. The result: dramatically better precision in the
final top-K, at a manageable cost because we only rerank a small candidate set.

## What You'll Learn

| Section | Topic |
|---------|-------|
| §1 | The Retrieval Quality Gap — why bi-encoders miss relevant documents |
| §2 | Bi-Encoder vs Cross-Encoder — architecture comparison |
| §3 | Why Reranking? — the two-stage retrieve-then-rerank pipeline |
| §4 | Cross-Encoder Architecture — how `[query, doc]` pairs produce relevance scores |
| §5 | Implementation from Scratch — FAISS → CrossEncoder → top-K |
| §6 | Score Analysis — score distributions and rank changes |
| §7 | Multiple Reranking Examples — 3+ queries showing rank improvements |
| §8 | Complete RAG Pipeline with Reranking |
| §9 | Latency Analysis — timing the speed-quality tradeoff |
| §10 | Synthesis — when reranking is worth the latency |

## Technical Stack

| Component | Model / Library |
|-----------|----------------|
| LLM | `Qwen/Qwen3-8B` (4-bit NF4 quantization) |
| Embeddings | `BAAI/bge-base-en-v1.5` via `sentence-transformers` |
| Reranker | `BAAI/bge-reranker-base` via `sentence-transformers` CrossEncoder |
| Vector Store | FAISS (faiss-cpu) |
| Cache | Google Drive `/content/drive/MyDrive/models` |

## Setup

Install all dependencies, mount Google Drive for model caching, and load the LLM,
embedding model, and cross-encoder reranker.

In [ ]:
# --- Install dependencies ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import torch, time, numpy as np
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# --- Load Qwen3-8B (4-bit NF4) ---
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=1024, temperature=0.7, do_sample=True):
    """Generate a response from Qwen3-8B given a chat-message list."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✅ LLM loaded: {MODEL_NAME}")

In [ ]:
# --- Load Embedding Model (bi-encoder) ---
from sentence_transformers import SentenceTransformer, CrossEncoder

embed_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR
)
print(f"✅ Bi-encoder loaded: BAAI/bge-base-en-v1.5  (dim={embed_model.get_sentence_embedding_dimension()})")

# --- Load Cross-Encoder Reranker ---
reranker = CrossEncoder("BAAI/bge-reranker-base")
print(f"✅ Cross-encoder loaded: BAAI/bge-reranker-base")

---
# §1 — The Retrieval Quality Gap

Modern RAG systems begin with a **first-stage retriever** — typically a bi-encoder that
represents queries and documents as fixed-size vectors and finds the nearest neighbours via
approximate nearest-neighbour search (ANN), usually FAISS.

This approach is **very fast** — FAISS can search millions of vectors in milliseconds. But
speed comes at a cost:

| Limitation | Why it happens |
|-----------|---------------|
| **Vocabulary mismatch** | The query says "heart attack" but the document says "myocardial infarction" — different tokens, distant embeddings |
| **Shallow interaction** | Query and document are encoded *independently* — no cross-attention between their tokens |
| **Compressed meaning** | A 768-dim vector must capture the *entire* semantics of a passage — subtle nuances are lost |
| **Keyword attraction** | Documents sharing surface words ("capital", "France") rank high even if they don't answer the question |

The result: the top-5 retrieved documents often contain only 1–2 that are truly relevant.
The rest is **noise** that the LLM must sift through — or worse, that misleads it.

### Concrete Example

Imagine a knowledge base about renewable energy. A query like *"What is the efficiency of
modern solar panels?"* will retrieve passages mentioning "solar" and "efficiency" broadly.
Passages about *solar thermal* efficiency or *historical* panel efficiency will score nearly
as high as the actually-relevant passage about *modern photovoltaic* efficiency.

We need a way to **re-assess** retrieved candidates with deeper understanding.

## Building the Synthetic Knowledge Base

We define a set of 20 passages on renewable energy, climate science, and technology. These
are hand-crafted to create *interesting reranking scenarios*: some passages share keywords
with likely queries but aren't actually relevant, while others are highly relevant but use
different vocabulary.

In [ ]:
# --- Synthetic Knowledge Base ---
knowledge_base = [
    # Solar panels — various subtopics
    "Modern monocrystalline silicon solar panels achieve 22-24% efficiency under standard test conditions. "
    "Recent advances in passivated emitter and rear contact (PERC) technology have pushed commercial "
    "module efficiency above 22%, with laboratory cells exceeding 26%.",

    "Solar thermal collectors concentrate sunlight to heat fluids, reaching temperatures above 400°C. "
    "Their thermal efficiency ranges from 30-70% depending on the collector type and operating temperature. "
    "These are distinct from photovoltaic panels that generate electricity directly.",

    "The cost of solar panels has dropped 99% since 1976, from $106 per watt to under $0.30 per watt. "
    "This dramatic price decline follows Swanson's Law, similar to Moore's Law for semiconductors.",

    "In the 1950s, Bell Labs created the first practical silicon solar cell with only 6% efficiency. "
    "Early space satellites used solar panels costing over $1,000 per watt. These historical panels "
    "bear little resemblance to today's high-efficiency modules.",

    # Wind energy
    "Modern wind turbines convert 35-45% of wind kinetic energy into electricity, approaching the "
    "theoretical Betz limit of 59.3%. Offshore wind farms achieve higher capacity factors (40-50%) "
    "than onshore installations (25-35%) due to stronger and more consistent wind patterns.",

    "Wind turbine blade design has evolved significantly. Current blades span over 100 meters and use "
    "composite materials including carbon fiber. Aerodynamic optimization via computational fluid "
    "dynamics has increased energy capture by 15-20% compared to older designs.",

    # Battery storage
    "Lithium-ion battery energy density has improved from 80 Wh/kg in 1991 to over 300 Wh/kg today. "
    "Grid-scale battery storage costs have fallen below $150/kWh, making renewable energy storage "
    "economically viable for the first time.",

    "Solid-state batteries replace liquid electrolytes with solid materials, promising 2-3x higher "
    "energy density, faster charging, and improved safety. Several manufacturers target commercial "
    "production by 2027, which could revolutionize electric vehicles and grid storage.",

    # Climate science
    "Global average surface temperature has risen approximately 1.1°C above pre-industrial levels. "
    "The rate of warming has accelerated, with the last decade being the warmest on record. "
    "Arctic regions are warming 2-3 times faster than the global average.",

    "Greenhouse gases trap infrared radiation in the atmosphere, creating a warming effect. "
    "CO2 concentrations have risen from 280 ppm pre-industrial to over 420 ppm today. "
    "Methane, though less abundant, has 80x the warming potential of CO2 over 20 years.",

    # Carbon capture
    "Direct air capture (DAC) technology removes CO2 directly from ambient air using chemical sorbents. "
    "Current DAC costs range from $250-600 per tonne of CO2, with the goal of reaching $100/tonne "
    "by 2030. Climeworks and Carbon Engineering are leading commercial operators.",

    "Ocean-based carbon sequestration uses enhanced weathering of minerals like olivine to accelerate "
    "natural CO2 absorption. Pilot projects show potential to sequester billions of tonnes annually, "
    "but ecological impacts on marine ecosystems require further study.",

    # Electric vehicles
    "Electric vehicle sales exceeded 14 million units globally in 2023, representing 18% of all "
    "new car sales. China leads with over 60% of global EV sales, followed by Europe at 25%.",

    "The average range of new electric vehicles has increased from 120 km in 2011 to over 400 km "
    "in 2024. Fast-charging networks now deliver 200+ km of range in 15 minutes, addressing "
    "the major consumer concern of range anxiety.",

    # Hydrogen fuel
    "Green hydrogen produced via electrolysis powered by renewable energy costs $4-6 per kg, compared "
    "to $1-2 per kg for grey hydrogen from natural gas. The cost gap is expected to narrow as "
    "electrolyzer technology improves and renewable electricity becomes cheaper.",

    # Nuclear energy
    "Small modular reactors (SMRs) generate 50-300 MW of power and can be factory-built and "
    "transported to site. Their modular design reduces construction time from 10+ years to 3-4 years. "
    "Several designs received regulatory approval in 2024.",

    # Energy policy
    "The Paris Agreement targets limiting global warming to 1.5°C above pre-industrial levels. "
    "Achieving this requires reaching net-zero CO2 emissions by 2050 and cutting global emissions "
    "43% below 2019 levels by 2030, according to IPCC assessments.",

    # Grid infrastructure
    "Smart grid technology uses digital communication to detect and react to local changes in usage. "
    "Advanced metering infrastructure (AMI) enables two-way communication between utilities and "
    "consumers, optimizing energy distribution and reducing waste by up to 15%.",

    # Bioenergy
    "Second-generation biofuels derived from agricultural waste and non-food crops avoid the food-vs-fuel "
    "debate. Cellulosic ethanol from switchgrass or corn stover can reduce lifecycle greenhouse gas "
    "emissions by 60-85% compared to gasoline.",

    # Geothermal
    "Enhanced geothermal systems (EGS) create artificial reservoirs by fracturing hot dry rock at "
    "depths of 3-10 km. Unlike conventional geothermal, EGS can be deployed almost anywhere, "
    "potentially providing 100+ GW of baseload power in the United States alone.",
]

print(f"📚 Knowledge base: {len(knowledge_base)} passages")
for i, doc in enumerate(knowledge_base[:3]):
    print(f"\n[{i}] {doc[:100]}...")
print(f"\n... and {len(knowledge_base) - 3} more passages.")

## Building the FAISS Index

We embed every passage with `bge-base-en-v1.5` and store the resulting vectors in a FAISS
flat (exact) index. For our 20-passage demo, exact search is instant; in production you'd
use `IndexIVFFlat` or `IndexHNSWFlat` for millions of vectors.

In [ ]:
import faiss

# Embed all passages
passage_embeddings = embed_model.encode(
    knowledge_base, normalize_embeddings=True, show_progress_bar=True
)
passage_embeddings = np.array(passage_embeddings, dtype="float32")

# Build FAISS index (Inner Product = cosine similarity when vectors are normalized)
dim = passage_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(passage_embeddings)

print(f"✅ FAISS index built: {faiss_index.ntotal} vectors, dimension={dim}")

---
# §2 — Bi-Encoder vs Cross-Encoder

Understanding the architectural difference between these two model types is the key to
understanding why reranking works.

## Bi-Encoder (used for first-stage retrieval)

```
┌─────────────┐   ┌─────────────┐
│  Query "Q"  │   │  Document "D"│
└──────┬──────┘   └──────┬──────┘
       │                 │
  ┌────▼────┐       ┌────▼────┐
  │ Encoder │       │ Encoder │      ← Same weights, but
  │  (BERT) │       │  (BERT) │        INDEPENDENT passes
  └────┬────┘       └────┬────┘
       │                 │
  ┌────▼────┐       ┌────▼────┐
  │ vec(Q)  │       │ vec(D)  │      ← Fixed-size vectors
  └────┬────┘       └────┬────┘
       │                 │
       └────►cosine◄─────┘           ← Simple similarity
              sim
```

**Key property**: Query and document never "see" each other's tokens. The similarity score
is a single dot product between two independently-computed vectors.

**Advantage**: Document vectors can be pre-computed and indexed → sub-millisecond search.

**Weakness**: No token-level interaction → can't detect nuanced relevance.

## Cross-Encoder (used for reranking)

```
┌──────────────────────────────┐
│    [CLS] Query [SEP] Doc    │     ← Concatenated input
└──────────────┬───────────────┘
               │
          ┌────▼────┐
          │  BERT   │               ← FULL cross-attention
          │ Encoder │                 between Q and D tokens
          └────┬────┘
               │
          ┌────▼────┐
          │ Linear  │               ← Classification head
          │  Layer  │
          └────┬────┘
               │
          relevance score            ← Single scalar
```

**Key property**: Every query token attends to every document token (and vice versa)
across all 12 transformer layers. This enables deep semantic understanding.

**Advantage**: Much more accurate relevance judgments.

**Weakness**: Must run a full forward pass for **every** (query, document) pair → O(N) cost.
Cannot pre-compute anything.

## Comparison Table

| Property | Bi-Encoder | Cross-Encoder |
|----------|-----------|---------------|
| Input | Q and D separately | [Q, D] concatenated |
| Interaction | None (late) | Full cross-attention (early) |
| Speed | ~1μs per comparison | ~10ms per pair |
| Pre-computation | ✅ Index offline | ❌ Must run online |
| Accuracy | Good | Excellent |
| Use case | Retrieve 100s | Rerank 10-50 |

### Demonstrating the Bi-Encoder Retrieval Quality Gap

Let's retrieve the top-10 passages for a specific query and examine whether the *truly*
relevant passage is at the top.

In [ ]:
def faiss_retrieve(query, k=10):
    """Retrieve top-k passages using bi-encoder + FAISS."""
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = faiss_index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({"index": int(idx), "score": float(score), "text": knowledge_base[idx]})
    return results

# Test query: specifically about modern solar panel efficiency
query = "What is the efficiency of modern solar panels?"
bi_results = faiss_retrieve(query, k=10)

print(f"Query: '{query}'")
print(f"{'='*80}")
for rank, r in enumerate(bi_results, 1):
    tag = "" 
    # The truly relevant passage is index 0 (PERC/monocrystalline)
    if r['index'] == 0:
        tag = " ✅ RELEVANT"
    elif r['index'] in [1, 3]:  # solar thermal, historical panels
        tag = " ⚠️ PARTIALLY RELATED"
    print(f"\nRank {rank:2d} | Bi-encoder score: {r['score']:.4f} | Passage [{r['index']}]{tag}")
    print(f"  {r['text'][:120]}...")

---
# §3 — Why Reranking?

The fundamental insight behind reranking is **the funnel metaphor**:

```
Full Corpus           →  First-Stage Retrieval  →   Reranking    →  Final Top-K
(1M+ documents)          (top-100, ~5ms)           (top-100→3,     (3 documents
                                                    ~200ms)         to the LLM)
```

### The two-stage pipeline

1. **Stage 1 — Recall-optimized retrieval** (bi-encoder + FAISS)
   - Goal: Don't miss any relevant document → retrieve broadly (top-20 or top-100)
   - Trade-off: Accept some false positives (noise) to maximize recall
   - Speed: Sub-millisecond on indexed vectors

2. **Stage 2 — Precision-optimized reranking** (cross-encoder)
   - Goal: From the candidate set, pick the *truly* relevant documents
   - Trade-off: Spend more time per candidate for much better accuracy
   - Speed: ~10ms per (query, document) pair → ~200ms for 20 candidates

### Why not just use the cross-encoder for everything?

Because cross-encoders are **O(N)** — they must evaluate every (query, doc) pair. For a
corpus of 1 million documents at 10ms each, that's **2.8 hours per query**. By using a
bi-encoder to narrow the field first, we get the best of both worlds:

- **Bi-encoder recall** ensures we don't miss relevant documents
- **Cross-encoder precision** ensures we surface the right ones

---
# §4 — Cross-Encoder Architecture Deep Dive

## How `BAAI/bge-reranker-base` Works

The BGE reranker is a BERT-like transformer fine-tuned for relevance classification:

### Input Formatting
```
Tokens:   [CLS]  What  is  the  efficiency  ...  [SEP]  Modern  mono  crystal  ...  [SEP]
Segment:    0      0   0   0       0        ...    0       1       1      1     ...    1
                ◄── query tokens ──►                    ◄── document tokens ──►
```

### Self-Attention Mechanism
In each of the 12 transformer layers, **every token attends to every other token**.
This means:
- "efficiency" in the query attends to "22-24%" in the document
- "modern" in the query attends to "PERC technology" in the document
- The model learns that "modern" + "PERC" + "22-24%" = **highly relevant**

### Output
The `[CLS]` token's final hidden state is passed through a linear layer to produce
a **single relevance score** (logit). Higher = more relevant.

### Training
The model was fine-tuned on the MS MARCO passage ranking dataset — millions of
(query, relevant passage, irrelevant passage) triplets — to learn what "relevance"
means across diverse topics.

### Key Insight: Why Scores Are Logits, Not Probabilities
Cross-encoder scores are raw logits (can be negative or >1). They're meaningful for
**ranking** (higher = better) but not for absolute thresholds. You compare scores within
a single query's candidate set, not across queries.

### Seeing the Cross-Encoder in Action

Let's score several (query, document) pairs to see how the cross-encoder differentiates
between truly relevant, partially relevant, and irrelevant passages.

In [ ]:
query = "What is the efficiency of modern solar panels?"

# Pick specific passages to compare
test_pairs = [
    (0, "Modern monocrystalline (PERC) — directly answers the question"),
    (1, "Solar thermal efficiency — related keyword, different topic"),
    (3, "1950s Bell Labs cells — historical, not modern"),
    (2, "Solar panel cost decline — mentions solar but not efficiency"),
    (4, "Wind turbine efficiency — different energy source entirely"),
    (8, "Global temperature rise — completely unrelated"),
]

pairs = [[query, knowledge_base[idx]] for idx, _ in test_pairs]
scores = reranker.predict(pairs)

print(f"Query: '{query}'")
print(f"{'='*90}")
print(f"{'Passage':55s} {'Score':>8s}  Verdict")
print(f"{'-'*90}")
for (idx, desc), score in sorted(zip(test_pairs, scores), key=lambda x: x[1], reverse=True):
    bar = '█' * int(max(0, (score + 5)) * 2)  # visual bar
    print(f"[{idx:2d}] {desc:50s}  {score:7.3f}  {bar}")

print(f"\n💡 Notice: The cross-encoder assigns a much higher score to the passage that")
print(f"   actually answers the question, even though others share surface keywords.")

---
# §5 — Reranking Implementation from Scratch

Now we'll build the complete two-stage retrieve-then-rerank pipeline:

1. **FAISS retrieval**: Get top-20 candidates (high recall, some noise)
2. **Cross-encoder reranking**: Score each candidate, sort by relevance
3. **Return top-3**: High-precision results for the LLM

We implement this step by step, with full visibility into scores and rank changes.

In [ ]:
def retrieve_and_rerank(query, initial_k=20, final_k=3, verbose=True):
    """
    Two-stage retrieval:
      Stage 1: FAISS bi-encoder retrieval (top initial_k)
      Stage 2: Cross-encoder reranking → top final_k
    Returns a dict with bi-encoder results, reranked results, and timing info.
    """
    # Stage 1: Bi-encoder retrieval
    t0 = time.time()
    q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
    bi_scores, bi_indices = faiss_index.search(q_emb, initial_k)
    t_retrieval = time.time() - t0

    bi_results = []
    for rank, (score, idx) in enumerate(zip(bi_scores[0], bi_indices[0])):
        bi_results.append({
            "bi_rank": rank + 1, "index": int(idx),
            "bi_score": float(score), "text": knowledge_base[idx]
        })

    # Stage 2: Cross-encoder reranking
    t0 = time.time()
    pairs = [[query, r["text"]] for r in bi_results]
    ce_scores = reranker.predict(pairs)
    t_rerank = time.time() - t0

    for r, ce_score in zip(bi_results, ce_scores):
        r["ce_score"] = float(ce_score)

    # Sort by cross-encoder score
    reranked = sorted(bi_results, key=lambda x: x["ce_score"], reverse=True)
    for new_rank, r in enumerate(reranked):
        r["rerank"] = new_rank + 1
        r["rank_change"] = r["bi_rank"] - r["rerank"]

    if verbose:
        print(f"Query: '{query}'")
        print(f"Stage 1 (bi-encoder): {initial_k} candidates in {t_retrieval*1000:.1f}ms")
        print(f"Stage 2 (cross-encoder): reranked in {t_rerank*1000:.1f}ms")
        print(f"\n{'Bi-Rank':>8s} → {'Rerank':>6s}  {'Δ':>4s}  {'Bi-Score':>9s}  {'CE-Score':>9s}  Passage")
        print(f"{'-'*100}")
        for r in reranked[:10]:  # show top-10 after reranking
            delta = r['rank_change']
            arrow = f"+{delta}" if delta > 0 else str(delta) if delta < 0 else " 0"
            marker = " ⬆️" if delta > 2 else " ⬇️" if delta < -2 else ""
            print(f"  #{r['bi_rank']:2d}    → #{r['rerank']:2d}   {arrow:>4s}  {r['bi_score']:9.4f}  {r['ce_score']:9.4f}  [{r['index']:2d}] {r['text'][:60]}...{marker}")

    return {
        "query": query,
        "bi_results": bi_results,
        "reranked": reranked[:final_k],
        "all_reranked": reranked,
        "t_retrieval_ms": t_retrieval * 1000,
        "t_rerank_ms": t_rerank * 1000,
    }

# Run it!
result1 = retrieve_and_rerank("What is the efficiency of modern solar panels?")

In [ ]:
print("\n" + "="*80)
print("FINAL TOP-3 AFTER RERANKING")
print("="*80)
for r in result1["reranked"]:
    print(f"\n🏆 Rank #{r['rerank']} (was bi-encoder #{r['bi_rank']}, Δ={r['rank_change']:+d})")
    print(f"   CE Score: {r['ce_score']:.4f} | Bi Score: {r['bi_score']:.4f}")
    print(f"   {r['text'][:200]}")

---
# §6 — Score Analysis

Let's analyze how reranking changes the score landscape. We'll look at:
1. **Score distributions** — bi-encoder vs cross-encoder
2. **Rank changes** — which passages move up/down the most
3. **Score correlation** — do bi-encoder and cross-encoder agree?

In [ ]:
# Analyze score distributions
bi_scores = [r["bi_score"] for r in result1["bi_results"]]
ce_scores = [r["ce_score"] for r in result1["bi_results"]]

print("SCORE DISTRIBUTION ANALYSIS")
print("="*60)
print(f"\nBi-encoder scores (cosine similarity):")
print(f"  Range:  [{min(bi_scores):.4f}, {max(bi_scores):.4f}]")
print(f"  Mean:   {np.mean(bi_scores):.4f}")
print(f"  Std:    {np.std(bi_scores):.4f}")
print(f"  Spread: {max(bi_scores) - min(bi_scores):.4f}")

print(f"\nCross-encoder scores (relevance logits):")
print(f"  Range:  [{min(ce_scores):.4f}, {max(ce_scores):.4f}]")
print(f"  Mean:   {np.mean(ce_scores):.4f}")
print(f"  Std:    {np.std(ce_scores):.4f}")
print(f"  Spread: {max(ce_scores) - min(ce_scores):.4f}")

print(f"\n💡 Key insight: Cross-encoder scores have a MUCH wider spread ({max(ce_scores)-min(ce_scores):.2f})")
print(f"   vs bi-encoder ({max(bi_scores)-min(bi_scores):.4f}). This means the cross-encoder is")
print(f"   much better at discriminating between relevant and irrelevant passages.")

In [ ]:
# Visualize rank changes as text-based chart
all_reranked = result1["all_reranked"]

print("RANK CHANGE VISUALIZATION")
print("="*80)
print(f"{'Passage':12s} {'Bi-Rank':>8s}  {'→':>3s}  {'Rerank':>6s}  {'Change':>7s}  Movement")
print(f"{'-'*80}")

for r in sorted(all_reranked, key=lambda x: abs(x['rank_change']), reverse=True):
    delta = r['rank_change']
    if delta > 0:
        bar = '▲' * min(delta, 15)
        label = f"↑{delta} (promoted)"
    elif delta < 0:
        bar = '▼' * min(abs(delta), 15)
        label = f"↓{abs(delta)} (demoted)"
    else:
        bar = '━'
        label = "unchanged"
    print(f"  [{r['index']:2d}]       #{r['bi_rank']:2d}    →   #{r['rerank']:2d}   {delta:+4d}     {bar} {label}")

In [ ]:
# Check Spearman rank correlation between bi-encoder and cross-encoder rankings
from collections import defaultdict

bi_ranks = [r["bi_rank"] for r in all_reranked]
ce_ranks = [r["rerank"] for r in all_reranked]

# Spearman correlation (manual calculation)
n = len(bi_ranks)
d_squared = sum((b - c) ** 2 for b, c in zip(bi_ranks, ce_ranks))
spearman = 1 - (6 * d_squared) / (n * (n**2 - 1))

print(f"Spearman rank correlation: {spearman:.4f}")
print(f"\nInterpretation:")
if spearman > 0.8:
    print(f"  Strong agreement — bi-encoder ranking is already quite good for this query")
elif spearman > 0.5:
    print(f"  Moderate agreement — reranking provides meaningful improvement")
else:
    print(f"  Weak agreement — reranking dramatically changes the ranking!")
    print(f"  This is exactly where reranking adds the most value.")

---
# §7 — Multiple Reranking Examples

Let's run reranking on several queries to see how consistently it improves results.
Each query is designed to test a different challenge:
1. **Vocabulary mismatch** — query uses different words than the relevant passage
2. **Keyword attraction** — multiple passages share surface keywords
3. **Specificity** — query asks for a specific detail buried in a broader topic

In [ ]:
# Example 2: Vocabulary mismatch — "removing CO2 from air" vs "Direct Air Capture"
print("\n" + "🔬 EXAMPLE 2: Vocabulary Mismatch".center(80, '='))
result2 = retrieve_and_rerank(
    "How can we remove carbon dioxide directly from the atmosphere?",
    initial_k=15, final_k=3
)
print(f"\n✅ Top result after reranking: passage [{result2['reranked'][0]['index']}]")
print(f"   Was bi-rank #{result2['reranked'][0]['bi_rank']} → now #1")

In [ ]:
# Example 3: Keyword attraction — many passages mention "energy"
print("\n" + "🔬 EXAMPLE 3: Keyword Attraction".center(80, '='))
result3 = retrieve_and_rerank(
    "What are the latest developments in battery storage for renewable energy?",
    initial_k=15, final_k=3
)
print(f"\n✅ Top results after reranking:")
for r in result3['reranked']:
    print(f"   [{r['index']:2d}] rank {r['bi_rank']}→{r['rerank']}  CE={r['ce_score']:.3f}  {r['text'][:80]}...")

In [ ]:
# Example 4: Specificity — asking about a precise fact
print("\n" + "🔬 EXAMPLE 4: Specificity".center(80, '='))
result4 = retrieve_and_rerank(
    "How much faster is Arctic warming compared to the global average?",
    initial_k=15, final_k=3
)
print(f"\n✅ Top results after reranking:")
for r in result4['reranked']:
    print(f"   [{r['index']:2d}] rank {r['bi_rank']}→{r['rerank']}  CE={r['ce_score']:.3f}  {r['text'][:80]}...")

In [ ]:
# Summary across all examples
print("RERANKING IMPROVEMENT SUMMARY")
print("="*80)
print(f"{'Query':55s} {'Top-1 Bi':>8s} {'Top-1 CE':>8s} {'Rank Δ':>7s}")
print(f"{'-'*80}")

for label, result in [("Modern solar panel efficiency", result1),
                       ("Remove CO2 from atmosphere", result2),
                       ("Battery storage for renewables", result3),
                       ("Arctic warming rate", result4)]:
    top = result['reranked'][0]
    orig_rank = top['bi_rank']
    delta = top['rank_change']
    print(f"  {label:53s} #{orig_rank:2d}       #1       {delta:+3d}")

print(f"\n💡 In every case, the cross-encoder promotes the most relevant passage to #1,")
print(f"   even when the bi-encoder ranked it lower.")

---
# §8 — Complete RAG Pipeline with Reranking

Now we combine everything into a full RAG pipeline:

```
User Query
    │
    ▼
┌─────────────────────────┐
│ Stage 1: FAISS Retrieval│ ── top-20 candidates
│ (bi-encoder, <5ms)      │
└──────────┬──────────────┘
           │
           ▼
┌─────────────────────────┐
│ Stage 2: Cross-Encoder  │ ── rerank to top-3
│ Reranking (~200ms)      │
└──────────┬──────────────┘
           │
           ▼
┌─────────────────────────┐
│ Stage 3: LLM Generation │ ── Qwen3-8B answers
│ with reranked context   │    using the best passages
└─────────────────────────┘
```

We'll compare the LLM's answer *with* and *without* reranking to see the impact on
generation quality.

In [ ]:
def rag_pipeline(query, use_reranking=True, initial_k=20, final_k=3):
    """
    Complete RAG pipeline:
      1. Retrieve candidates via FAISS
      2. Optionally rerank with cross-encoder
      3. Generate answer with Qwen3-8B
    """
    t_start = time.time()

    if use_reranking:
        result = retrieve_and_rerank(query, initial_k=initial_k, final_k=final_k, verbose=False)
        context_docs = result["reranked"]
        t_retrieval = result["t_retrieval_ms"]
        t_rerank = result["t_rerank_ms"]
    else:
        # No reranking — just use top-K from bi-encoder directly
        candidates = faiss_retrieve(query, k=final_k)
        context_docs = [{"text": c["text"], "index": c["index"], "bi_score": c["score"]} for c in candidates]
        t_retrieval = 0  # not timed separately here
        t_rerank = 0

    # Build context string
    context = "\n\n".join(
        f"[Document {i+1}]: {doc['text']}" for i, doc in enumerate(context_docs)
    )

    # Generate with Qwen3-8B
    messages = [
        {"role": "system", "content": (
            "You are a knowledgeable assistant. Answer the user's question based ONLY on the "
            "provided context documents. If the context doesn't contain the answer, say so. "
            "Be specific and cite document numbers when relevant."
        )},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
    ]

    t_gen_start = time.time()
    answer = generate(messages, max_new_tokens=512, temperature=0.3)
    t_gen = (time.time() - t_gen_start) * 1000

    t_total = (time.time() - t_start) * 1000

    return {
        "query": query,
        "answer": answer,
        "context_docs": context_docs,
        "use_reranking": use_reranking,
        "t_retrieval_ms": t_retrieval,
        "t_rerank_ms": t_rerank,
        "t_generation_ms": t_gen,
        "t_total_ms": t_total,
    }

print("✅ RAG pipeline defined")

In [ ]:
# RAG WITH reranking
query = "What is the efficiency of modern solar panels and how has it improved?"

print("RAG WITH RERANKING")
print("="*80)
result_with = rag_pipeline(query, use_reranking=True)

print(f"\nRetrieved passages (after reranking):")
for i, doc in enumerate(result_with["context_docs"]):
    print(f"  [{doc['index']}] {doc['text'][:100]}...")

print(f"\n📝 Answer:\n{result_with['answer']}")
print(f"\n⏱️  Retrieval: {result_with['t_retrieval_ms']:.1f}ms | Reranking: {result_with['t_rerank_ms']:.1f}ms | Generation: {result_with['t_generation_ms']:.1f}ms")

In [ ]:
# RAG WITHOUT reranking (baseline)
print("RAG WITHOUT RERANKING (Baseline)")
print("="*80)
result_without = rag_pipeline(query, use_reranking=False)

print(f"\nRetrieved passages (bi-encoder top-3 only):")
for i, doc in enumerate(result_without["context_docs"]):
    print(f"  [{doc['index']}] {doc['text'][:100]}...")

print(f"\n📝 Answer:\n{result_without['answer']}")
print(f"\n⏱️  Generation: {result_without['t_generation_ms']:.1f}ms (no reranking step)")

In [ ]:
# Side-by-side comparison
print("SIDE-BY-SIDE COMPARISON")
print("="*80)
print(f"\nQuery: {query}")

print(f"\n{'─'*40} WITH RERANKING {'─'*40}")
print(f"Passages used: {[doc['index'] for doc in result_with['context_docs']]}")
print(f"Answer (first 300 chars): {result_with['answer'][:300]}...")

print(f"\n{'─'*40} WITHOUT RERANKING {'─'*37}")
print(f"Passages used: {[doc['index'] for doc in result_without['context_docs']]}")
print(f"Answer (first 300 chars): {result_without['answer'][:300]}...")

print(f"\n💡 The reranked pipeline provides more specific, accurate context to the LLM,")
print(f"   resulting in a better-grounded answer.")

### Another RAG Comparison: Battery Technology

In [ ]:
query2 = "What is the current state of solid-state battery technology?"

print("RAG COMPARISON: Solid-State Batteries")
print("="*80)

r_with = rag_pipeline(query2, use_reranking=True)
r_without = rag_pipeline(query2, use_reranking=False)

print(f"\nWith reranking — passages: {[d['index'] for d in r_with['context_docs']]}")
print(f"Answer: {r_with['answer'][:400]}")

print(f"\nWithout reranking — passages: {[d['index'] for d in r_without['context_docs']]}")
print(f"Answer: {r_without['answer'][:400]}")

---
# §9 — Latency Analysis

Reranking adds latency. Is it worth it? Let's measure precisely:

| Stage | What happens | Expected latency |
|-------|-------------|------------------|
| Embedding query | Encode 1 query with bi-encoder | 1-5ms |
| FAISS search | Nearest-neighbour lookup | <1ms |
| Cross-encoder reranking | Score N (query, doc) pairs | 5-15ms × N |
| LLM generation | Qwen3-8B generates answer | 1-10s |

The key question: does 100-300ms of reranking (for 20 candidates) meaningfully
improve the answer quality, given that LLM generation takes 1-10 seconds anyway?

In [ ]:
# Latency benchmark
import time

query = "What are the environmental impacts of green hydrogen production?"
n_trials = 3

# Benchmark: Bi-encoder encoding
times_embed = []
for _ in range(n_trials):
    t0 = time.time()
    _ = embed_model.encode([query], normalize_embeddings=True)
    times_embed.append((time.time() - t0) * 1000)

# Benchmark: FAISS search
q_emb = embed_model.encode([query], normalize_embeddings=True).astype("float32")
times_faiss = []
for _ in range(n_trials):
    t0 = time.time()
    _ = faiss_index.search(q_emb, 20)
    times_faiss.append((time.time() - t0) * 1000)

# Benchmark: Cross-encoder reranking for different candidate set sizes
candidates = faiss_retrieve(query, k=20)
times_rerank = {}
for k in [5, 10, 15, 20]:
    pairs = [[query, c["text"]] for c in candidates[:k]]
    trial_times = []
    for _ in range(n_trials):
        t0 = time.time()
        _ = reranker.predict(pairs)
        trial_times.append((time.time() - t0) * 1000)
    times_rerank[k] = np.mean(trial_times)

print("LATENCY BENCHMARK (averaged over 3 trials)")
print("="*60)
print(f"\n📊 Query embedding (bi-encoder):  {np.mean(times_embed):7.1f}ms")
print(f"📊 FAISS search (20 candidates):  {np.mean(times_faiss):7.2f}ms")
print(f"\n📊 Cross-encoder reranking:")
for k, t in times_rerank.items():
    bar = '█' * int(t / 10)
    print(f"   {k:2d} candidates: {t:7.1f}ms  {bar}")

print(f"\n📊 Per-candidate reranking cost: ~{times_rerank[20]/20:.1f}ms")
print(f"\n💡 Total pipeline overhead for 20-candidate reranking: ~{np.mean(times_embed) + np.mean(times_faiss) + times_rerank[20]:.0f}ms")
print(f"   Compare this to LLM generation time of 1-10 seconds — reranking is cheap!")

In [ ]:
# Analyze the speed-quality tradeoff
print("SPEED vs QUALITY TRADEOFF")
print("="*60)
print(f"\n{'Candidates':>12s}  {'Rerank Time':>12s}  {'% of 3s Gen':>12s}  Recommendation")
print(f"{'-'*60}")

gen_time_ms = 3000  # typical LLM generation time
for k in [5, 10, 15, 20]:
    t = times_rerank[k]
    pct = (t / gen_time_ms) * 100
    if pct < 3:
        rec = "✅ Negligible overhead"
    elif pct < 10:
        rec = "✅ Good tradeoff"
    elif pct < 20:
        rec = "⚠️ Noticeable but acceptable"
    else:
        rec = "❌ Consider reducing K"
    print(f"  {k:5d}        {t:8.1f}ms      {pct:6.1f}%     {rec}")

print(f"\n💡 Typical sweet spot: rerank 15-20 candidates, which adds <10% to total latency.")

---
# §10 — Synthesis: When Is Reranking Worth It?

## Decision Framework

| Factor | Favors Reranking | Favors No Reranking |
|--------|-----------------|--------------------|
| **Corpus ambiguity** | Many similar passages on different subtopics | Passages are clearly distinct |
| **Query complexity** | Multi-faceted or nuanced queries | Simple keyword lookups |
| **Latency budget** | >200ms acceptable for retrieval | Ultra-low latency required (<50ms) |
| **Candidate set** | 10-50 candidates | >100 candidates (too expensive) |
| **Downstream task** | LLM generation (seconds) dominates latency | Retrieval-only (no LLM) |

## Batch Reranking Strategies

For production systems handling high query volumes:

1. **Batch scoring**: Accumulate multiple (query, doc) pairs and score them in a single
   forward pass through the cross-encoder. GPU batch processing is 3-5× faster than
   sequential scoring.

2. **Tiered reranking**: Use a lightweight cross-encoder (e.g., MiniLM-based) to rerank
   top-100 → top-20, then a heavier model (e.g., bge-reranker-large) for top-20 → top-3.

3. **Caching**: Cache cross-encoder scores for frequently-seen (query, document) pairs.
   Particularly effective when the document corpus is relatively static.

4. **Async prefetching**: Start reranking before the user finishes typing (on partial
   queries), then refine when the full query arrives.

## What We Demonstrated

| Concept | Key Takeaway |
|---------|-------------|
| Retrieval Quality Gap | Bi-encoders miss nuanced relevance — top-5 often has 2-3 irrelevant passages |
| Bi-Encoder vs Cross-Encoder | Independent encoding (fast) vs joint encoding (accurate) |
| Two-Stage Pipeline | Broad recall → precise reranking → best of both worlds |
| Score Analysis | Cross-encoder scores have 10-50× wider spread → better discrimination |
| Latency | Reranking 20 candidates adds <10% to total RAG latency |
| RAG Quality | Reranked context leads to more specific, better-grounded LLM answers |

### Batch Reranking Demo

Let's demonstrate that batch scoring is significantly faster than sequential scoring.

In [ ]:
# Batch vs sequential reranking speed comparison
query = "What renewable energy storage solutions exist?"
candidates = faiss_retrieve(query, k=20)
pairs = [[query, c["text"]] for c in candidates]

# Sequential: score one pair at a time
t0 = time.time()
sequential_scores = []
for pair in pairs:
    score = reranker.predict([pair])
    sequential_scores.append(score[0])
t_sequential = (time.time() - t0) * 1000

# Batch: score all pairs at once
t0 = time.time()
batch_scores = reranker.predict(pairs)
t_batch = (time.time() - t0) * 1000

speedup = t_sequential / t_batch

print("BATCH vs SEQUENTIAL RERANKING")
print("="*50)
print(f"Sequential (20 × 1 pair):  {t_sequential:7.1f}ms")
print(f"Batch      (1 × 20 pairs): {t_batch:7.1f}ms")
print(f"Speedup:                    {speedup:.1f}×")
print(f"\n💡 Batch scoring is {speedup:.1f}× faster. In production, always batch!")
print(f"\nScores match: {np.allclose(sequential_scores, batch_scores, atol=1e-4)}")

### Final End-to-End Example

One final complete RAG query to demonstrate the full pipeline in action.

In [ ]:
final_query = "What are small modular reactors and when will they be available?"

print("FINAL END-TO-END RAG WITH RERANKING")
print("="*80)

# Show the retrieval + reranking step
rr = retrieve_and_rerank(final_query, initial_k=15, final_k=3)

# Generate final answer
final_result = rag_pipeline(final_query, use_reranking=True, initial_k=15, final_k=3)

print(f"\n{'─'*80}")
print(f"📝 FINAL ANSWER:")
print(f"{'─'*80}")
print(final_result['answer'])
print(f"\n⏱️  Total pipeline: {final_result['t_total_ms']:.0f}ms")
print(f"   (Retrieval: {final_result['t_retrieval_ms']:.1f}ms + Reranking: {final_result['t_rerank_ms']:.1f}ms + Generation: {final_result['t_generation_ms']:.0f}ms)")

---
# Conclusion

Cross-encoder reranking is one of the most impactful, lowest-effort improvements
you can add to a RAG system:

- **3 lines of code** to add reranking to an existing pipeline
- **<10% latency overhead** relative to LLM generation time
- **Dramatic precision improvement** in the final context window

The two-stage retrieve-then-rerank pattern is standard in production search systems
(Google, Bing, e-commerce) and translates directly to RAG. If you're building a RAG
system and not using reranking, you're leaving significant quality on the table.

### Next Steps
- Try `BAAI/bge-reranker-v2-m3` for multilingual reranking
- Experiment with different `initial_k` and `final_k` ratios
- Combine reranking with other techniques: HyDE, fusion retrieval, query expansion
- For very large candidate sets, explore distilled/lighter rerankers